# Non-Local LLM Inference




In [ ]:
# import packages used in this script
# built-in packages
import os
import textwrap
import time

# pypi packages
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
from pydantic import BaseModel
from enum import Enum


# own packages
import start

# Non-Local Inference
Non-local calls are one way to interact with large language models with Python. 

You send a request over the internet to a provider's servers (e.g., OpenAI, Anthropic, Google) and receive a generated response back. 

These calls are made via application programming interface (API), which allow the provider to verify who is making the request (aka who to bill for it). That's where API keys come in; they're the credential that authenticates each request. 

Colab Option A) Save API key to Secrets

Navigate to Secrets in the left-hand panel, create a variable called `OPENAI_API_KEY`. ADD SCREENSHOT


Local Option B) Save API key to .env

- Create a file called `.env` in your working directory (see the template in 00_secrets.txt)
- Ensure that `.env` is in your `.gitignore` file
- Add your keys to `.env`:
   ```
   OPENAI_API_KEY = "sk-..."
   ANTHROPIC_API_KEY = "sk-ant-..."
   GOOGLE_API_KEY = "AQ..."
   ```

In [ ]:
# check that the api key loaded to the environment
# if loaded, prints the key
# if not loaded, prints ERROR
key = "OPENAI_API_KEY"
print(os.environ.get(key, f"ERROR: Variable {key} Not Found"))

## ⚙️ Model Settings

We'll use GPT 5.6 Terra. Here's a link to information about this model:
- https://developers.openai.com/api/docs/models/gpt-5.6-terra

Note: The 1,050,000 context window means it can hold over 1 million tokens in the context window.



In [ ]:
MODEL = "gpt-5.6-terra" 
IN_RATE = 2.00
OUT_RATE = 12.00

The API call can take two types of prompts, a system promp and a user prompt. 

**System Prompt**

System prompts (also called System messages) are persistent instructions for how the model should behave. Once set, they guide every subsequent interaction.

**User Prompt**

User prompts are

In [ ]:
system_prompt = "You are a researcher in educational psychology."
user_prompt = "What is a dialogic prompt?"

Other settings we can adjust are:

**Temperature**

A parameter (ranging from 0-2) that controls the randomness of text generated by LLMs during inference. Each token is assigned a probability of occurance based on the prompt and the tokens that come before it. Temperature modifies this probability distribution such that at higher temperatures increase the likelihood of selecting less probable tokens.


**Top P**

An alternative to sampling with temperature, called nucleus sampling, where the model considers the results of the tokens with `top_p` probability mass. So 0.1 means only the tokens comprising the top 10% probability mass are considered.

**Token Limits**

The maximum number of tokens output by the model.

For a complete list see the openai.chat.completions.create() documentation: https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create

## Prompt gpt 5.6 terra

In [ ]:
# initialize the model
openai = OpenAI()

In [ ]:
# format prompt
PROMPT = [
    #{"role": "system", "content": CONTEXT},
    {"role": "user", "system": system_prompt, "content": user_prompt}
  ]

# model settings
client = openai.chat.completions.create(
    model = MODEL, 
    messages = PROMPT
    )

response = client.choices[0].message.content
print(response)

### Classify a Dialogic Prompt


In [ ]:
CASE = "Okay, and what would you do?"

PROMPT = f"""Classify the following utterance as yes or no based on whether it is a dialogic prompt. 
A dialogic prompt is defined as an utterance that implies, encourages, requests, or expects a 
new speaker (or multiple new speakers) to make a verbal contribution.\n
Here is the utterance: 
\n\"\"\"{CASE}\"\"\"\n
Output format:\n
- First, give a brief explanation (1 short sentence, under 20 words) 
justifying your answer.
- Then give your final answer as "yes" or "no"."""

print(textwrap.fill(PROMPT, width = 92))


In [ ]:
# model settings
client = openai.chat.completions.create(
    model = MODEL, 
    messages = [{"role": "user", "content": PROMPT}]
    )

response = client.choices[0].message.content
print(response)

### Classify a Dialogic Prompt with the LLM Codebook

[ADD screenshot of codebook]

In [ ]:
# load the prompt codebook
path_to_prompts = start.CODEBOOK_DIR / "llm_prompt_codebook.xlsx"
prompts = pd.read_excel(path_to_prompts)

In [ ]:
CASE = "Okay, and what would you do?"

In [ ]:
PROMPT = (prompts.loc[prompts.id == "Coding1", "prompt"].item() + " " +
          prompts.loc[prompts.id == "Construct", "prompt"].item() + " " +
          prompts.loc[prompts.id == "Prompt1", "prompt"].item() + " " +
          f"\n\"\"\"{CASE}\"\"\"\n" + 
          prompts.loc[prompts.id == "Format1", "prompt"].item()
          )

print(PROMPT)

In [ ]:
# format prompt

# model settings
client = openai.chat.completions.create(
    model = MODEL, 
    messages = [{"role": "user", "content": PROMPT}]
    )

response = client.choices[0].message.content

print(response)

### Classify Many Dialogic Prompts with the LLM Codebook

In [ ]:
# import data
DATA_SOURCE = "train"

path_to_data = start.DATA_DIR / f"{DATA_SOURCE}.xlsx"
df = pd.read_excel(path_to_data)

In [ ]:
SAMPLE_SIZE = 5

sample = df.sample(n = SAMPLE_SIZE, ignore_index = True)

In [ ]:
for row in sample.index:
    print(row)

In [ ]:
for row in sample.index:
    CASE = sample.loc[row, "text"]

    # construct prompt w case
    PROMPT = (prompts.loc[prompts.id == "Coding1", "prompt"].item() + " " +
              prompts.loc[prompts.id == "Construct", "prompt"].item() + " " +
              prompts.loc[prompts.id == "Prompt1", "prompt"].item() + " " +
              f"\n\"\"\"{CASE}\"\"\"\n" + 
              prompts.loc[prompts.id == "Format1", "prompt"].item()
              )

    client = openai.chat.completions.create(
        model = MODEL, 
        messages = [{"role": "user", "content": PROMPT}]
        )

    response = client.choices[0].message.content
    print(f"Utterance: \"{CASE}\"")
    print(f"Response: {response}\n")
    

#### 📋 Formatting Output
Force the model to respond in a specific format. In this case we want to separate the explanation from the classification (yes or no)

Instead of `client.chat.completions.creat()` we'll use `client.chat.completions.parse()`.

In [ ]:
# format output
class Answer(str, Enum):
    yes = "yes"
    no = "no"

class Classification(BaseModel):
    explanation: str  
    label: Answer

In [ ]:
for row in sample.index:
    CASE = sample.loc[row, "text"]

    # construct prompt w case
    PROMPT = (prompts.loc[prompts.id == "Coding1", "prompt"].item() + " " +
              prompts.loc[prompts.id == "Construct", "prompt"].item() + " " +
              prompts.loc[prompts.id == "Prompt1", "prompt"].item() + " " +
              f"\n\"\"\"{CASE}\"\"\"\n" + 
              prompts.loc[prompts.id == "Format1", "prompt"].item()
              )

    try:
        client = openai.chat.completions.parse(
            model = MODEL,
            messages = [{"role": "user", "content": PROMPT}],
            response_format = Classification
            ) 
        parsed = client.choices[0].message.parsed
    except Exception as e:
        print(f"Row {row} failed: {e}")
        parsed = None
        prompt_gpt = None

    if parsed is None:
        label = None
    else:
        label = parsed.label

    if label == "yes":
        response_code = 1
    elif label == "no":
        response_code = 0
    else:
        response_code = None
        print(f"Unexpected response at row {row}: {label!r}")

    print(f"Utterance: \"{CASE}\"")
    print(f"LLM Response: {response_code}")
    print(f"Human Response: {sample.loc[row, "code_human"]}\n")


#### 💰 Calculating Cost + Runtime


In [ ]:
MODEL = "gpt-5.6-terra" 

#SAMPLE_SIZE = 30
#sample = df.sample(n = SAMPLE_SIZE, ignore_index = True)

IN_RATE = 2.00
OUT_RATE = 12.00

In [ ]:
# initialize output objects
tokens_in_column = []
tokens_out_column = []

total_input_cost = None
total_output_cost = None

code_gpt = []
explanation_gpt = []

tp = 0
tn = 0
fp = 0
fn = 0

In [ ]:
start_time = time.perf_counter()

In [ ]:
for row in sample.index:
    CASE = sample.loc[row, "text"]

    # construct prompt w case
    PROMPT = (prompts.loc[prompts.id == "Coding1", "prompt"].item() + " " +
              prompts.loc[prompts.id == "Construct", "prompt"].item() + " " +
              prompts.loc[prompts.id == "Prompt1", "prompt"].item() + " " +
              f"\n\"\"\"{CASE}\"\"\"\n" + 
              prompts.loc[prompts.id == "Format1", "prompt"].item()
              )

    try:
        client = openai.chat.completions.parse(
            model = MODEL,
            messages = [{"role": "user", "content": PROMPT}],
            response_format = Classification
            ) 
        parsed = client.choices[0].message.parsed
    except Exception as e:
        print(f"Row {row} failed: {e}")
        parsed = None
        client = None

    tokens_input = client.usage.prompt_tokens

    if parsed is None:
        label = None
        tokens_output = 0
    else:
        label = parsed.label
        tokens_output = client.usage.completion_tokens

    tokens_in_column.append(tokens_input)
    tokens_out_column.append(tokens_output)

    if total_input_cost is None:
        total_input_cost = tokens_input / 1_000_000 * IN_RATE
        total_output_cost = tokens_output / 1_000_000 * OUT_RATE
    else:
        total_input_cost = total_input_cost + (tokens_input / 1_000_000 * IN_RATE)
        total_output_cost = total_output_cost + (tokens_output / 1_000_000 * OUT_RATE)

    if label == "yes":
        response_code = 1
    elif label == "no":
        response_code = 0
    else:
        response_code = None
        print(f"Unexpected response at row {row}: {label!r}")

    code_gpt.append(response_code)
    explanation_gpt.append(parsed.explanation if parsed else None)
    
    if df.loc[row, "code_human"] == 1 and response_code == 1:
        tp = tp + 1
    elif df.loc[row, "code_human"] == 0 and response_code == 0:
        tn = tn + 1
    elif df.loc[row, "code_human"] == 0 and response_code == 1:
        fp = fp + 1
    elif df.loc[row, "code_human"] == 1 and response_code == 0:
        fn = fn + 1
    print(f"Utterance: \"{CASE}\"")
    print(f"LLM Response: {response_code}")
    print(f"Human Response: {sample.loc[row, "code_human"]}\n")

# end runtime counter
end_time = time.perf_counter()

In [ ]:
sample[f"code_{MODEL}"] = code_gpt
sample[f"explanation_{MODEL}"] = explanation_gpt

In [ ]:
# ---- save the results ----
# classifications
results_data_file = f"{DATA_SOURCE}_{MODEL}.xlsx"
path_to_data_results = start.RESULTS_DIR / "non-local" / results_data_file

sample.to_excel(path_to_data_results, index = False)

# model performance
path_to_model_results = start.RESULTS_DIR / "classifications.xlsx"
results = pd.read_excel(path_to_model_results, sheet_name = f"{DATA_SOURCE}")

new_row = {"model": MODEL,
           "utterances": sample.shape[0],
           "tp": tp,
           "tn": tn,
           "fp": fp,
           "fn": fn,
           "cost": total_input_cost + total_output_cost,
           "runtime": end_time - start_time
           }

results = pd.concat([results, pd.DataFrame([new_row])], ignore_index = True)

with pd.ExcelWriter(
    path_to_model_results, engine = "openpyxl", mode = "a", if_sheet_exists = "replace") as writer:
    results.to_excel(writer, sheet_name=f"{DATA_SOURCE}", index = False)